In [1]:
import pandas as pd
from transformers import AutoTokenizer, RobertaModel, AutoConfig
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
import torch

import utils
import const
import models

# Read and Prepare Data

In [2]:
relations_df = utils.get_relations()
relations_df.head()

,docid,verb1,verb2,eiid1,eiid2,relation
0,WSJ_20130322_159,apologized,happened,E1,E5,VAGUE
1,WSJ_20130322_159,apologized,wrapped,E1,E6,BEFORE
2,WSJ_20130322_159,apologized,seemed,E1,E10,BEFORE
3,WSJ_20130322_159,apologized,yield,E1,E11,VAGUE
4,WSJ_20130322_159,happened,wrapped,E5,E6,BEFORE


In [3]:
unique_docids = relations_df['docid'].unique()

docs_df = utils.get_docs(unique_docids)
docs_df.head()

,raw_text,text,sentences
docid,,,
WSJ_20130322_159,Israeli Prime Minister Benjamin Netanyahu <EVE...,Israeli Prime Minister Benjamin Netanyahu [E1]...,[Israeli Prime Minister Benjamin Netanyahu [E1...
nyt_20130322_strange_computer,"Our <TIMEX3 type=""DATE"" value=""PRESENT_REF"" ti...",Our [TIMEX3]digital[/TIMEX3] age is all about ...,[Our [TIMEX3]digital[/TIMEX3] age is all about...
CNN_20130321_821,"Barack Obama would <EVENT class=""OCCURRENCE"" e...",Barack Obama would [E1]make[/E1] a great stand...,[Barack Obama would [E1]make[/E1] a great stan...
nyt_20130321_cyprus,"A Cyprus <EVENT class=""OCCURRENCE"" eid=""e2001""...",A Cyprus [E2001]exit[/E2001] from the euro uni...,[A Cyprus [E2001]exit[/E2001] from the euro un...
bbc_20130322_1353,"Israel's prime minister has <EVENT class=""OCCU...",Israel's prime minister has [E1]apologised[/E1...,[Israel's prime minister has [E1]apologised[/E...


In [4]:
relations_df = utils.create_context_windows(relations_df, docs_df)
relations_df = utils.create_relation_labels(relations_df)
relations_df.head()

,docid,verb1,verb2,eiid1,eiid2,relation,context_window,relation_id
0,WSJ_20130322_159,apologized,happened,E1,E5,VAGUE,Israeli Prime Minister Benjamin Netanyahu [T1]...,0
1,WSJ_20130322_159,apologized,wrapped,E1,E6,BEFORE,Israeli Prime Minister Benjamin Netanyahu [T1]...,1
2,WSJ_20130322_159,apologized,seemed,E1,E10,BEFORE,Israeli Prime Minister Benjamin Netanyahu [T1]...,1
3,WSJ_20130322_159,apologized,yield,E1,E11,VAGUE,Israeli Prime Minister Benjamin Netanyahu [T1]...,0
4,WSJ_20130322_159,happened,wrapped,E5,E6,BEFORE,Israeli Prime Minister Benjamin Netanyahu apol...,1


# Prepare Data for Training

In [5]:
tokenizer = models.create_temp_rel_tokenizer()

In [6]:
test_docid = 'CNN_20130322_314'

test_df = relations_df[relations_df['docid'] == test_docid]
temp_df = relations_df[relations_df['docid'] != test_docid]

train_df, val_df = train_test_split(
    temp_df,
    test_size=0.05,
    random_state=42,
    stratify=temp_df["relation_id"]
)

val_df = utils.augment_data(val_df)

print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

Train: 6841
Validation: 722
Test: 39


In [7]:
train_loader = utils.create_data_loader(train_df, tokenizer, batch_size=16)
val_loader = utils.create_data_loader(val_df, tokenizer, batch_size=16)
test_loader = utils.create_data_loader(test_df, tokenizer, batch_size=16)

print("num training batches:", len(train_loader))

input_ids shape: torch.Size([6841, 512])
attention_mask shape: torch.Size([6841, 512])
labels shape: torch.Size([6841])
input_ids shape: torch.Size([722, 321])
attention_mask shape: torch.Size([722, 321])
labels shape: torch.Size([722])
input_ids shape: torch.Size([39, 162])
attention_mask shape: torch.Size([39, 162])
labels shape: torch.Size([39])
num training batches: 428


# Model Creation

In [8]:
loss_fn = torch.nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0

    all_preds = []
    all_labels = []

    for input_ids, attention_mask, labels in loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)  # (B,) long

        # Keep only samples that contain both target markers
        has_t1 = (input_ids == model.t1_id).any(dim=1)
        has_t2 = (input_ids == model.t2_id).any(dim=1)
        valid = has_t1 & has_t2
        if not valid.any():
            continue

        input_ids = input_ids[valid]
        attention_mask = attention_mask[valid]
        labels = labels[valid]

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask)  # (B, C)
        loss = loss_fn(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * input_ids.size(0)
        all_preds.append(torch.argmax(logits, dim=-1).cpu())
        all_labels.append(labels.cpu())

    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()

    macro_f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)

    return total_loss / len(loader.dataset), macro_f1, acc

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0

    all_preds = []
    all_labels = []

    for input_ids, attention_mask, labels in loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        # Keep only samples that contain both target markers
        has_t1 = (input_ids == model.t1_id).any(dim=1)
        has_t2 = (input_ids == model.t2_id).any(dim=1)
        valid = has_t1 & has_t2
        if not valid.any():
            continue

        input_ids = input_ids[valid]
        attention_mask = attention_mask[valid]
        labels = labels[valid]

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(logits, labels)

        preds = torch.argmax(logits, dim=-1)

        total_loss += loss.item() * input_ids.size(0)
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()

    macro_f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)

    return {
        "val_loss": total_loss / len(loader.dataset),
        "macro_f1": macro_f1,
        "accuracy": acc,
    }

In [9]:
# Build model
num_labels = len(const.relation2id)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.TemporalRelationsModel(num_labels=num_labels, tokenizer=tokenizer).to(device)

# Cross-entropy setup
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To dis

# Training

In [10]:
# Train
epochs = 12
for epoch in range(epochs):
    train_loss, train_f1, train_acc = train_one_epoch(model, train_loader, optimizer, device)

    val_metrics = evaluate(model, val_loader, device)
    val_loss = val_metrics["val_loss"]
    val_f1 = val_metrics["macro_f1"]
    val_acc = val_metrics["accuracy"]

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}")

Epoch 1/12 | Train Loss: 0.8926 | Train F1: 0.3384 | Train Acc: 0.6404 | Val Loss: 0.6788 | Val F1: 0.4021 | Val Acc: 0.7493
Epoch 2/12 | Train Loss: 0.6350 | Train F1: 0.4369 | Train Acc: 0.7753 | Val Loss: 0.6190 | Val F1: 0.4471 | Val Acc: 0.7716
Epoch 3/12 | Train Loss: 0.4877 | Train F1: 0.5175 | Train Acc: 0.8240 | Val Loss: 0.5924 | Val F1: 0.4997 | Val Acc: 0.7911
Epoch 4/12 | Train Loss: 0.4009 | Train F1: 0.5955 | Train Acc: 0.8529 | Val Loss: 0.5933 | Val F1: 0.5342 | Val Acc: 0.7855
Epoch 5/12 | Train Loss: 0.3234 | Train F1: 0.6683 | Train Acc: 0.8838 | Val Loss: 0.6191 | Val F1: 0.5053 | Val Acc: 0.7827
Epoch 6/12 | Train Loss: 0.2383 | Train F1: 0.7653 | Train Acc: 0.9174 | Val Loss: 0.6639 | Val F1: 0.5573 | Val Acc: 0.7827
Epoch 7/12 | Train Loss: 0.1949 | Train F1: 0.8299 | Train Acc: 0.9332 | Val Loss: 0.6939 | Val F1: 0.5411 | Val Acc: 0.7883
Epoch 8/12 | Train Loss: 0.1396 | Train F1: 0.8869 | Train Acc: 0.9531 | Val Loss: 0.6960 | Val F1: 0.5887 | Val Acc: 0.7967


# Analysis

In [11]:
from sklearn.metrics import confusion_matrix

# Predict on test set
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for input_ids, attention_mask, labels in test_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        # Keep only samples that contain both target markers
        has_t1 = (input_ids == model.t1_id).any(dim=1)
        has_t2 = (input_ids == model.t2_id).any(dim=1)
        valid = has_t1 & has_t2
        if not valid.any():
            continue

        input_ids = input_ids[valid]
        attention_mask = attention_mask[valid]
        labels = labels[valid]
        
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(logits, dim=-1).cpu()

        all_preds.append(preds)
        all_true.append(labels)

y_pred = torch.cat(all_preds).numpy()
y_true = torch.cat(all_true).detach().cpu().numpy()

# Confusion matrix (rows=true, cols=pred)
label_ids = [0, 1, 2, 3]
cm = confusion_matrix(y_true, y_pred, labels=label_ids)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{const.id2relation[i]}" for i in label_ids],
    columns=[f"pred_{const.id2relation[i]}" for i in label_ids],
)
display(cm_df)

test_metrics = evaluate(model, test_loader, device)
print(f"Test Loss: {test_metrics['val_loss']:.4f} | Test F1: {test_metrics['macro_f1']:.4f} | Test Acc: {test_metrics['accuracy']:.4f}")

,pred_VAGUE,pred_BEFORE,pred_AFTER,pred_EQUAL
true_VAGUE,2,6,3,0
true_BEFORE,2,13,0,0
true_AFTER,2,1,7,0
true_EQUAL,1,0,2,0


Test Loss: 2.8342 | Test F1: 0.4004 | Test Acc: 0.5641


In [13]:
before_id = const.relation2id["BEFORE"]
after_id = const.relation2id["AFTER"]

mask = pd.Series(y_true).isin([before_id, after_id]).values
y_true_ba = y_true[mask]
y_pred_ba = y_pred[mask]

cm_ba_df = pd.crosstab(
    pd.Series([const.id2relation[i] for i in y_true_ba], name="true"),
    pd.Series([const.id2relation[i] for i in y_pred_ba], name="pred"),
    dropna=False
)

acc_ba = accuracy_score(y_true_ba, y_pred_ba)

display(cm_ba_df)
print(f"Accuracy (true BEFORE/AFTER only): {acc_ba:.4f}")

pred,AFTER,BEFORE,VAGUE
true,,,
AFTER,7,1,2
BEFORE,0,13,2


Accuracy (true BEFORE/AFTER only): 0.8000


# Saving model weights

In [12]:
filename = 'temp_rel_roberta.pt'
torch.save(model.state_dict(), filename)
print(f"Model saved to '{filename}'")

Model saved to 'temp_rel_roberta.pt'


# Test with loaded weights

In [ ]:
tokenizer_test = models.create_temp_rel_tokenizer()

In [28]:
from importlib import reload


reload(utils)

<module 'utils' from '/home/anton/Dev/event_extraction/utils.py'>

In [29]:
test_df = relations_df[relations_df['docid'] == 'CNN_20130322_314']
test_loader = utils.create_data_loader(test_df, tokenizer_test, batch_size=16, shuffle=False)

input_ids shape: torch.Size([39, 162])
attention_mask shape: torch.Size([39, 162])
labels shape: torch.Size([39])


In [30]:
state_dict = torch.load('./temp_rel_roberta.pt')

model_test = models.TemporalRelationsModel(num_labels=num_labels, tokenizer=tokenizer_test)
model_test.load_state_dict(state_dict, strict=False)
model_test = model_test.to(device)
model_test.eval()

all_preds, all_true = [], []

with torch.no_grad():
    for input_ids, attention_mask, labels in test_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        # Keep only samples that contain both target markers
        has_t1 = (input_ids == model_test.t1_id).any(dim=1)
        has_t2 = (input_ids == model_test.t2_id).any(dim=1)
        valid = has_t1 & has_t2
        if not valid.any():
            continue

        input_ids = input_ids[valid]
        attention_mask = attention_mask[valid]
        labels = labels[valid]
        
        logits = model_test(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(logits, dim=-1).cpu()

        all_preds.append(preds)
        all_true.append(labels)

y_pred = torch.cat(all_preds).numpy()
y_true = torch.cat(all_true).detach().cpu().numpy()

# Confusion matrix (rows=true, cols=pred)
label_ids = [0, 1, 2, 3]
cm = confusion_matrix(y_true, y_pred, labels=label_ids)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{const.id2relation[i]}" for i in label_ids],
    columns=[f"pred_{const.id2relation[i]}" for i in label_ids],
)
display(cm_df)

test_metrics = evaluate(model_test, test_loader, device)
print(f"Test Loss: {test_metrics['val_loss']:.4f} | Test F1: {test_metrics['macro_f1']:.4f} | Test Acc: {test_metrics['accuracy']:.4f}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,pred_VAGUE,pred_BEFORE,pred_AFTER,pred_EQUAL
true_VAGUE,10,0,1,0
true_BEFORE,0,15,0,0
true_AFTER,0,0,10,0
true_EQUAL,0,1,0,2


Test Loss: 0.4290 | Test F1: 0.9181 | Test Acc: 0.9487


In [31]:
test_df["predicted_relation"] = [const.id2relation[pred] for pred in y_pred]

In [32]:
test_df.head(10)

,docid,verb1,verb2,eiid1,eiid2,relation,context_window,relation_id,predicted_relation
780,CNN_20130322_314,arrived,scoring,E1,E2,AFTER,President Barack Obama [T1]arrived[/T1] in ref...,2,AFTER
781,CNN_20130322_314,arrived,leaving,E1,E4,VAGUE,President Barack Obama [T1]arrived[/T1] in ref...,0,VAGUE
782,CNN_20130322_314,arrived,apologized,E1,E5,VAGUE,President Barack Obama [T1]arrived[/T1] in ref...,0,VAGUE
783,CNN_20130322_314,arrived,killed,E1,E7,AFTER,President Barack Obama [T1]arrived[/T1] in ref...,2,AFTER
784,CNN_20130322_314,scoring,leaving,E2,E4,BEFORE,President Barack Obama arrived in refugee-floo...,1,BEFORE
785,CNN_20130322_314,scoring,apologized,E2,E5,BEFORE,President Barack Obama arrived in refugee-floo...,1,BEFORE
786,CNN_20130322_314,scoring,killed,E2,E7,AFTER,President Barack Obama arrived in refugee-floo...,2,AFTER
787,CNN_20130322_314,leaving,apologized,E4,E5,EQUAL,President Barack Obama arrived in refugee-floo...,3,EQUAL
788,CNN_20130322_314,leaving,killed,E4,E7,AFTER,President Barack Obama arrived in refugee-floo...,2,AFTER
789,CNN_20130322_314,apologized,killed,E5,E7,AFTER,President Barack Obama arrived in refugee-floo...,2,AFTER
